In [2]:
import pandas as pd

df = pd.read_csv(r'C:\Users\USER\Desktop\machine_learning\ai_fraud_detector\ai-fraud-detector\backend\data\raw\email_dataset.csv')

# Look at the balance: Are there more scams than real emails?
print(df['label'].value_counts())

# Let's find the most common "scam" words
from collections import Counter
scams = df[df['label'] == 1]['text'].str.cat(sep=' ').lower().split()
common_scam_words = Counter(scams).most_common(10)
print(f"Top Scam Words: {common_scam_words}")

label
1    6000
0    4000
Name: count, dtype: int64
Top Scam Words: [('your', 10177), ('to', 9520), ('the', 8835), ('keywords:', 6000), ('and', 4664), ('is', 4500), ('a', 3826), ('this', 3500), ('account', 3294), ('you', 3171)]


In [3]:
import pandas as pd
from collections import Counter
# We use Scikit-learn's built-in list of stop words
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

df = pd.read_csv("../data/raw/email_dataset.csv")

# 1. Get all words from scam emails
scams = df[df['label'] == 1]['text'].str.cat(sep=' ').lower().split()

# 2. Filter: Only keep the word if it's NOT in the stop words list
# Also, let's make sure the word is at least 3 characters long to avoid 're', 'fw', etc.
custom_stops = ENGLISH_STOP_WORDS.union(['keywords:', 'subject:', 'email'])
filtered_scams = [w for w in scams if w not in custom_stops]

# 3. Re-run the Counter
common_scam_words = Counter(filtered_scams).most_common(20)

print(f"Cleaned Top Scam Words: {common_scam_words}")

Cleaned Top Scam Words: [('account', 3294), ('account.', 2500), ('security', 2466), ('dear', 1896), ('verification', 1809), ('desk,', 1719), ('team,', 1593), ('link', 1513), ('follow', 1500), ('avoid', 1500), ('soon.', 1419), ('payment', 1419), ('password', 1293), ('restore', 1247), ('access', 1226), ('respond', 1170), ('notice', 1117), ('hotel', 1098), ('detected', 1050), ('notice,', 1031)]


In [4]:
from collections import Counter
import pandas as pd

common_scam_df = pd.DataFrame(common_scam_words, columns=['word', 'counts'])
print(common_scam_df)

            word  counts
0        account    3294
1       account.    2500
2       security    2466
3           dear    1896
4   verification    1809
5          desk,    1719
6          team,    1593
7           link    1513
8         follow    1500
9          avoid    1500
10         soon.    1419
11       payment    1419
12      password    1293
13       restore    1247
14        access    1226
15       respond    1170
16        notice    1117
17         hotel    1098
18      detected    1050
19       notice,    1031


In [6]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import classification_report, confusion_matrix

# 1. Load the Data
# Ensure you are pointing to the correct path from your notebook folder
df = pd.read_csv("../data/raw/email_dataset.csv")

# 2. Quick Look
print(f"Dataset Shape: {df.shape}")
df.head()

Dataset Shape: (10000, 5)


,text,label,phishing_type,severity,confidence
0,Subject: Office maintenance\n\nThanks for your...,0,legitimate,low,0.95
1,"Hello, your profile has been locked. Use the s...",1,credential_harvesting,high,0.89
2,"Hi there, congratulations! You are the winner ...",1,financial_scam,medium,0.69
3,"Attention, this is the fraud prevention accoun...",1,authority_scam,high,0.91
4,"Notice, your profile has been restricted. Use ...",1,credential_harvesting,high,0.80


In [7]:
# Check for nulls and the balance of classes
print(df['label'].value_counts(normalize=True)) 

print("--" * 40)

# Visualizing a Scam vs. a Safe message
print("--- SCAM EXAMPLE ---")
print(df[df['label'] == 1]['text'].iloc[0])

print("--" * 40)

print("\n--- SAFE EXAMPLE ---")
print(df[df['label'] == 0]['text'].iloc[0])

label
1    0.6
0    0.4
Name: proportion, dtype: float64
--------------------------------------------------------------------------------
--- SCAM EXAMPLE ---
Hello, your profile has been locked. Use the secure link to verify your username and restore access. Enter your verification code to continue.

Keywords: pin update password sign in settings password

Sincerely,
Riley Khan
--------------------------------------------------------------------------------

--- SAFE EXAMPLE ---
Subject: Office maintenance

Thanks for your help on the analysis. I've pushed the changes and left comments in the PR for clarity.

Let me know if the timing works for you.

Best,
Avery Kim


Since our dataset is relatively balanced (60% scam emails and 40% safe emails), accuracy can still be used as a performance metric. However, if the dataset were imbalanced, relying solely on accuracy would be insufficient to properly evaluate the model.

In [8]:
# Load the tokenizer and model
model_ckpt = "mshenoda/roberta-spam"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)
model = AutoModelForSequenceClassification.from_pretrained(model_ckpt)

def predict_scam(text):
    # Prepare the text for the model (Tokenization)
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    
    # Get the "Logits" (Raw scores before becoming probabilities)
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Convert Logits to Probabilities using Softmax
    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    
    # Get the highest probability
    conf, classes = torch.max(probs, dim=-1)
    
    return "SCAM" if classes.item() == 1 else "SAFE", conf.item()

# TEST THE VIBE
print(predict_scam("Thanks for your help on the analysis. I've pushed the changes and left comments in the PR for clarity."))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: mshenoda/roberta-spam
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


('SAFE', 0.9999983310699463)


In [9]:
# Sample 100 rows to test speed and accuracy
test_df = df.sample(300)
test_df['pred'], test_df['confidence'] = zip(*test_df['text'].apply(predict_scam))

# Note: You'll need to map your dataset labels (0, 1) to (SAFE, SCAM) to compare
y_true = test_df['label'].map({0: 'SAFE', 1: 'SCAM'})
print(classification_report(y_true, test_df['pred']))

              precision    recall  f1-score   support

        SAFE       0.42      1.00      0.59       110
        SCAM       1.00      0.19      0.32       190

    accuracy                           0.49       300
   macro avg       0.71      0.59      0.45       300
weighted avg       0.79      0.49      0.42       300



## fine tune the pretrained model

In [10]:
def predict_scam_tuned(text, threshold=0.15): # Lower threshold = More sensitive
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    
    # Get probabilities
    probs = torch.nn.functional.softmax(outputs.logits, dim=-1)
    
    # Extract the probability for index 1 (SCAM)
    # [Row 0, Column 1]
    scam_probability = probs[0][1].item()
    
    # If there is even a 15% chance it's a scam, we label it SCAM
    label = "SCAM" if scam_probability > threshold else "SAFE"
    
    return label, scam_probability

# Apply the tuned function
test_df_ft = df.sample(300)
test_df_ft['pred'], test_df_ft['score'] = zip(*test_df['text'].apply(lambda x: predict_scam_tuned(x, threshold=0.000002)))

In [11]:
test_df_ft

,text,label,phishing_type,severity,confidence,pred,score
2327,"Account holder, this is the cia accounts team....",1,authority_scam,high,0.86,SCAM,0.000010
7990,Subject: Classroom logistics\n\nThanks for you...,0,legitimate,low,0.98,SCAM,0.000051
8419,"Dear customer, this is an urgent notice regard...",1,urgency,medium,0.89,SAFE,0.000002
8282,"Hello, your account has been restricted. Use t...",1,credential_harvesting,high,0.83,SCAM,0.000003
3898,Subject: Quarterly roadmap\n\nPlease find the ...,0,legitimate,low,0.99,SCAM,0.000162
...,...,...,...,...,...,...,...
7462,"Subject: Quarterly roadmap\n\nHello, confirmin...",0,legitimate,low,0.95,SCAM,0.000011
2509,"Notice, your profile has been restricted. Use ...",1,credential_harvesting,high,0.82,SCAM,0.000004
8777,"Notice, we detected unusual activity on your a...",1,social_engineering,high,0.94,SCAM,0.000009
9469,Subject: Lab availability\n\nSharing the revis...,0,legitimate,low,0.99,SAFE,0.000002


In [12]:
y_true_ft = test_df_ft['label'].map({0: 'SAFE', 1: 'SCAM'})
print(classification_report(y_true_ft, test_df['pred']))

              precision    recall  f1-score   support

        SAFE       0.41      0.87      0.55       123
        SCAM       0.56      0.11      0.19       177

    accuracy                           0.42       300
   macro avg       0.48      0.49      0.37       300
weighted avg       0.49      0.42      0.34       300



In [13]:
import pandas as pd
from datasets import Dataset

# 1. Convert your CSV to a Hugging Face Dataset
# Assuming 'text' and 'label' are your column names
ds = Dataset.from_pandas(df[['text', 'label']])

# 2. Split into Train and Test (80% / 20%)
ds = ds.train_test_split(test_size=0.2)

# 3. Tokenize the data
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True)

tokenized_datasets = ds.map(tokenize_function, batched=True)

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

In [16]:
from transformers import TrainingArguments, Trainer
import numpy as np
import evaluate

metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

training_args = TrainingArguments(
    output_dir="fraud_detector_checkpoints",
    eval_strategy="epoch",      # Check accuracy after every round
    learning_rate=2e-5,               # Very small steps so we don't 'break' the brain
    per_device_train_batch_size=8,    # How many emails to look at at once
    num_train_epochs=3,               # Go through the dataset 3 times
    weight_decay=0.01,
    logging_dir='./logs',
    push_to_hub=False,
)

small_train_dataset = tokenized_datasets["train"].shuffle(seed=42).select(range(500))
small_eval_dataset = tokenized_datasets["test"].shuffle(seed=42).select(range(100))

# Pass these to the trainer instead
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=small_train_dataset,
    eval_dataset=small_eval_dataset,
    compute_metrics=compute_metrics,
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [17]:
# START THE BRAIN REWIRING
trainer.train()

# Save the newly improved model
trainer.save_model("./my_custom_fraud_model")
tokenizer.save_pretrained("./my_custom_fraud_model")

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 